In [2]:
import sqlite3

# Directory where the repository is cloned
dir = '/Users/ljob/Desktop/cnbs-predictor-1/data/'

# Path to input CFS forecast database
cfs_database = dir + 'input/cfs_forecast_data.db'

In [ ]:
# Deleting old forecasts

# Cutoff value for cfs_run
cutoff_cfs_run = 2024080100

# Connect to the database
conn = sqlite3.connect(cfs_database)
cursor = conn.cursor()

# Check how many rows will be deleted (optional but safe)
cursor.execute("""
    SELECT COUNT(*) FROM cfs_forecast_data WHERE cfs_run < ?
""", (cutoff_cfs_run,))
rows_to_delete = cursor.fetchone()[0]

print(f"Number of rows to delete: {rows_to_delete}")

# Only delete if there are rows to delete
if rows_to_delete > 0:
    # Perform the deletion
    cursor.execute("""
        DELETE FROM cfs_forecast_data WHERE cfs_run < ?
    """, (cutoff_cfs_run,))

    # Commit changes
    conn.commit()
    print(f"Deleted {rows_to_delete} rows.")
else:
    print("No rows to delete.")

# Close the connection
conn.close()

In [ ]:
# Changing the name of a variable

# Connect to the database
conn = sqlite3.connect(cfs_database)
cursor = conn.cursor()

# Rename the column
cursor.execute("""
    ALTER TABLE cfs_forecast_data RENAME COLUMN value TO 'value [mm]';
""")

# Commit the change
conn.commit()

# Close the connection
conn.close()

print("Column renamed from 'value' to 'value [mm]'.")

Column renamed from 'value' to 'cvalue [mm]'.


In [5]:
# Deleting a table

# Connect to the database
conn = sqlite3.connect('/Users/ljob/Desktop/cfs_forecast_data.db')
cursor = conn.cursor()

# Drop the table if it exists
cursor.execute("""
    DROP TABLE IF EXISTS forecast_data;
""")

# Commit the changes
conn.commit()

# Close the connection
conn.close()

print("Table 'forecast_data' has been removed.")

Table 'forecast_data' has been removed.


In [6]:
import sqlite3
import pandas as pd

# Database and table
db_path = "/Users/ljob/Desktop/cfs_forecast_data.db"
table_name = "cfs_forecast_data"

# Connect to the database
conn = sqlite3.connect(db_path)

# Read the table
df = pd.read_sql(f"SELECT * FROM {table_name}", conn)

# Create forecast_month in YYYY-MM format
df["forecast_month"] = (
    df["year"].astype(int).astype(str)
    + "-"
    + df["month"].astype(int).astype(str).str.zfill(2)
)

# Drop the old columns
df = df.drop(columns=["year", "month"])

# Overwrite the table
df.to_sql(table_name, conn, if_exists="replace", index=False)

conn.close()

print("Successfully updated table.")

Successfully updated table.


In [9]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('/Users/ljob/Desktop/cfs_forecast_data.db')

# Read the table
df = pd.read_sql("SELECT * FROM cfs_forecast_data", conn)

# Reorder the columns
df = df[
    [
        "cfs_run",
        "forecast_month",
        "lake",
        "surface_type",
        "component",
        "value"
    ]
]

# Replace the table with the reordered version
df.to_sql("cfs_forecast_data", conn, if_exists="replace", index=False)

conn.close()